In [4]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
warnings.filterwarnings('ignore')

In [15]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# # metadata 
# print(bank_marketing.metadata) 
  
# # variable information 
# print(bank_marketing.variables) 


### 2. CẤU HÌNH

In [16]:
SEED          = 42
DROP_DURATION = True
TEST_SIZE     = 0.2
# N_TUNE        = 4000
N_TUNE        = None

CV            = 3
SCORING       = 'f1'
N_JOBS        = 4

# ---- Không gian tìm kiếm 4 siêu tham số ----
LOGC          = (-2, 2)                      # C     in [1e-2, 1e2]
LOGG          = (-3, 0)                      # gamma in [1e-3, 1e0]
KERNELS       = ['rbf', 'poly', 'sigmoid']
DEGREES       = [2, 3, 4]

# ---- Ngân sách (số lần đánh giá) cho mỗi phương pháp ----
GRID_C, GRID_G = 3, 3
BUDGET         = GRID_C * GRID_G * len(KERNELS) * len(DEGREES)   # 4*4*3*3 = 144
PSO_PARTICLES  = 9
PSO_ITERS      = BUDGET // PSO_PARTICLES                       # 12 x 12 = 144
N_RUNS         = 5
print('Budget:', BUDGET, '| PSO:', PSO_PARTICLES, 'particles x', PSO_ITERS, 'iterations')

Budget: 81 | PSO: 9 particles x 9 iterations


### 3. TIỀN XỬ LÝ DỮ LIỆU ###

In [17]:
X = X.copy()
y = y.iloc[:, 0].astype(str).str.strip().map({'yes': 1, 'no': 0}).astype(int)
if DROP_DURATION and 'duration' in X.columns:
    X = X.drop(columns='duration')

cat_cols = X.select_dtypes(exclude='number').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()
X[cat_cols] = X[cat_cols].fillna('unknown')

print('Numeric    :', num_cols)
print('Categorical:', cat_cols)
print('Tỉ lệ lớp dương (yes): %.3f' % y.mean())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

# Train tập con
# n_tune = min(N_TUNE, len(X_train))
# X_tune, _, y_tune, _ = train_test_split(
#     X_train, y_train, train_size=n_tune, stratify=y_train, random_state=SEED)
# print('Train:', X_train.shape, '| Test:', X_test.shape, '| Tune subset:', X_tune.shape)

# Mới: train toàn bộ 
if N_TUNE is None or N_TUNE >= len(X_train):
    X_tune, y_tune = X_train, y_train
else:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=N_TUNE, stratify=y_train, random_state=SEED)
print('Train:', X_train.shape, '| Test:', X_test.shape, '| Tune:', X_tune.shape)

skf = StratifiedKFold(n_splits=CV, shuffle=True, random_state=SEED)

def make_pipe(C=1.0, gamma='scale', kernel='rbf', degree=3):
    pre = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
    svm = SVC(C=C, gamma=gamma, kernel=kernel, degree=degree, class_weight='balanced',
              cache_size=500, max_iter=200000, random_state=SEED)   # max_iter chặn trường hợp poly hội tụ quá chậm
    return Pipeline([('pre', pre), ('svm', svm)])

def params_to_points(params_list):
    # mỗi cấu hình -> [log10 C, log10 gamma, chỉ số kernel, degree]
    return np.array([[np.log10(p['svm__C']), np.log10(p['svm__gamma']),
                      KERNELS.index(p['svm__kernel']), p['svm__degree']] for p in params_list], dtype=float)

def pack(best_params, cv_score, t, res):
    return {'C': float(best_params['svm__C']), 'gamma': float(best_params['svm__gamma']),
            'kernel': best_params['svm__kernel'], 'degree': int(best_params['svm__degree']),
            'cv_score': float(cv_score), 'time': t, 'n_evals': len(res['params']),
            'history': np.maximum.accumulate(res['mean_test_score']),
            'points': params_to_points(res['params']), 'scores': np.asarray(res['mean_test_score'])}

Numeric    : ['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous']
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Tỉ lệ lớp dương (yes): 0.117
Train: (36168, 15) | Test: (9043, 15) | Tune: (36168, 15)


### 4. GRID SEARCH

In [ ]:
t0 = time.time()
grid = GridSearchCV(
    make_pipe(),
    param_grid={'svm__C': np.logspace(LOGC[0], LOGC[1], GRID_C),
                'svm__gamma': np.logspace(LOGG[0], LOGG[1], GRID_G),
                'svm__kernel': KERNELS,
                'svm__degree': DEGREES},
    cv=skf, scoring=SCORING, n_jobs=N_JOBS, pre_dispatch='n_jobs', refit=False)
grid.fit(X_tune, y_tune)
grid_res = pack(grid.best_params_, grid.best_score_, time.time() - t0, grid.cv_results_)
print('Grid  |', {k: grid_res[k] for k in ['C', 'gamma', 'kernel', 'degree']},
      '| CV-F1=%.4f | %.1fs | %d evals' % (grid_res['cv_score'], grid_res['time'], grid_res['n_evals']))

### 5. RANDOM SEARCH (lặp `N_RUNS` lần với seed khác nhau)

In [ ]:
rand_runs = []
for r in range(N_RUNS):
    t0 = time.time()
    rs = RandomizedSearchCV(
        make_pipe(),
        param_distributions={'svm__C': loguniform(10**LOGC[0], 10**LOGC[1]),
                             'svm__gamma': loguniform(10**LOGG[0], 10**LOGG[1]),
                             'svm__kernel': KERNELS,
                             'svm__degree': DEGREES},
        n_iter=BUDGET, cv=skf, scoring=SCORING, n_jobs=N_JOBS, random_state=SEED + r, refit=False)
    rs.fit(X_tune, y_tune)
    rand_runs.append(pack(rs.best_params_, rs.best_score_, time.time() - t0, rs.cv_results_))
    b = rand_runs[-1]
    print('Random run %d | C=%.4g gamma=%.4g kernel=%s degree=%d | CV-F1=%.4f | %.1fs' %
          (r, b['C'], b['gamma'], b['kernel'], b['degree'], b['cv_score'], b['time']))

### 6. PSO (Cài đặt thủ công cho không gian hỗn hợp)
Mỗi hạt là một vector 4 chiều liên tục: [log10(C), log10(gamma), chỉ số kernel, degree]. Khi đánh giá, hai chiều rời rạc được làm tròn xuống (floor) thành kernel / degree cụ thể. Cập nhật vận tốc: v = w·v + c1·r1·(pbest − x) + c2·r2·(gbest − x), với w giảm tuyến tính 0.9 → 0.4.

In [ ]:
LO = np.array([LOGC[0], LOGG[0], 0.0, float(DEGREES[0])])
HI = np.array([LOGC[1], LOGG[1], float(len(KERNELS)), float(DEGREES[-1] + 1)])
EPS = 1e-9   # để floor không vượt chỉ số cuối

def decode(p):
    k = int(min(np.floor(p[2]), len(KERNELS) - 1))
    d = int(min(np.floor(p[3]), DEGREES[-1]))
    return 10 ** p[0], 10 ** p[1], KERNELS[k], d

def objective(p):
    C, g, k, d = decode(p)
    return cross_val_score(make_pipe(C, g, k, d), X_tune, y_tune,
                           cv=skf, scoring=SCORING, n_jobs=N_JOBS).mean()

def pso(n_particles, n_iter, seed, w_max=0.9, w_min=0.4, c1=1.5, c2=1.5):
    rng = np.random.default_rng(seed)
    vmax = 0.2 * (HI - LO)
    pos = rng.uniform(LO, HI - EPS, (n_particles, 4))
    vel = rng.uniform(-vmax, vmax, (n_particles, 4))
    all_pts, all_scores = [], []

    def evaluate(P):
        s = np.array([objective(p) for p in P])
        for p in P:
            C, g, k, d = decode(p)
            all_pts.append([np.log10(C), np.log10(g), KERNELS.index(k), d])
        all_scores.extend(s)
        return s

    pbest, pbest_s = pos.copy(), evaluate(pos)
    g_i = pbest_s.argmax(); gbest, gbest_s = pbest[g_i].copy(), pbest_s[g_i]

    for it in range(1, n_iter):
        w = w_max - (w_max - w_min) * it / max(n_iter - 1, 1)
        r1, r2 = rng.random((n_particles, 4)), rng.random((n_particles, 4))
        vel = np.clip(w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos), -vmax, vmax)
        pos = np.clip(pos + vel, LO, HI - EPS)
        s = evaluate(pos)
        better = s > pbest_s
        pbest[better], pbest_s[better] = pos[better], s[better]
        if pbest_s.max() > gbest_s:
            g_i = pbest_s.argmax(); gbest, gbest_s = pbest[g_i].copy(), pbest_s[g_i]

    return gbest, gbest_s, np.array(all_pts, dtype=float), np.array(all_scores)

pso_runs = []
for r in range(N_RUNS):
    t0 = time.time()
    gbest, gbest_s, pts, scs = pso(PSO_PARTICLES, PSO_ITERS, seed=SEED + r)
    C, g, k, d = decode(gbest)
    pso_runs.append({'C': C, 'gamma': g, 'kernel': k, 'degree': d, 'cv_score': float(gbest_s),
                     'time': time.time() - t0, 'n_evals': len(scs),
                     'history': np.maximum.accumulate(scs), 'points': pts, 'scores': scs})
    print('PSO run %d    | C=%.4g gamma=%.4g kernel=%s degree=%d | CV-F1=%.4f | %.1fs | %d evals' %
          (r, C, g, k, d, gbest_s, pso_runs[-1]['time'], len(scs)))

### 7. Tổng hợp kết quả tuning (CV trên tập tuning)

In [ ]:
def summarize(name, runs):
    cv = [r['cv_score'] for r in runs]; tm = [r['time'] for r in runs]
    return {'Method': name, 'n_evals': runs[0]['n_evals'],
            'CV-F1 mean': np.mean(cv), 'CV-F1 std': np.std(cv), 'CV-F1 best': np.max(cv),
            'Time mean (s)': np.mean(tm), 'Time std (s)': np.std(tm)}

tuning_df = pd.DataFrame([summarize('Grid Search', [grid_res]),
                          summarize('Random Search', rand_runs),
                          summarize('PSO', pso_runs)]).round(4)
tuning_df

In [ ]:
# Cấu hình tốt nhất của từng lần chạy
rows = [{'Method': 'Grid Search', 'run': 0, **{k: grid_res[k] for k in ['C', 'gamma', 'kernel', 'degree', 'cv_score']}}]
for name, runs in [('Random Search', rand_runs), ('PSO', pso_runs)]:
    for i, r in enumerate(runs):
        rows.append({'Method': name, 'run': i, **{k: r[k] for k in ['C', 'gamma', 'kernel', 'degree', 'cv_score']}})
pd.DataFrame(rows).round(5)

In [ ]:
from scipy.stats import mannwhitneyu

rand_scores = [r['cv_score'] for r in rand_runs]
pso_scores  = [r['cv_score'] for r in pso_runs]

# PSO có cao hơn Random không? (một phía)
stat, p_greater = mannwhitneyu(pso_scores, rand_scores, alternative='greater')
# Hai nhóm có khác nhau không? (hai phía)
_,    p_two     = mannwhitneyu(pso_scores, rand_scores, alternative='two-sided')

print(f'PSO    : mean={np.mean(pso_scores):.4f}  std={np.std(pso_scores, ddof=1):.4f}  (n={len(pso_scores)})')
print(f'Random : mean={np.mean(rand_scores):.4f}  std={np.std(rand_scores, ddof=1):.4f}  (n={len(rand_scores)})')
print(f'Mann-Whitney U = {stat:.1f}')
print(f'p-value (PSO > Random, một phía) = {p_greater:.4f}')
print(f'p-value (khác nhau, hai phía)    = {p_two:.4f}')
print('Có ý nghĩa thống kê ở mức 0.05' if p_two < 0.05 else 'Chưa có ý nghĩa thống kê ở mức 0.05')

# Grid Search chỉ có 1 giá trị (tất định) nên không kiểm định được, chỉ so sánh trực tiếp:
print(f'\\nGrid Search (1 giá trị): {grid_res["cv_score"]:.4f}')
print(f'Số lần chạy của PSO vượt Grid   : {sum(s > grid_res["cv_score"] for s in pso_scores)}/{len(pso_scores)}')
print(f'Số lần chạy của Random vượt Grid: {sum(s > grid_res["cv_score"] for s in rand_scores)}/{len(rand_scores)}')

### *** CHỌN LẠI TỪ TOP-K CẤU HÌNH ***

In [ ]:
N_RECHECK = 20000    # tập lớn hơn để kiểm tra lại
TOP_K = 3

n_re = min(N_RECHECK, len(X_train))
X_re, _, y_re, _ = train_test_split(
    X_train, y_train, train_size=n_re, stratify=y_train, random_state=SEED + 1)

def recheck(run, name):
    # lấy top-k cấu hình khác nhau theo CV-F1 trên tập tuning
    order = np.argsort(run['scores'])[::-1]
    seen, cands = set(), []
    for i in order:
        lc, lg, ki, d = run['points'][i]
        key = (round(lc, 4), round(lg, 4), int(ki), int(d))
        if key in seen: continue
        seen.add(key); cands.append((10**lc, 10**lg, KERNELS[int(ki)], int(d)))
        if len(cands) == TOP_K: break
    best = None
    for C, g, k, d in cands:
        s = cross_val_score(make_pipe(C, g, k, d), X_re, y_re,
                            cv=skf, scoring=SCORING, n_jobs=N_JOBS).mean()
        print(f'{name}: C={C:.4g} gamma={g:.4g} kernel={k} degree={d} | CV-F1 (n={n_re}) = {s:.4f}')
        if best is None or s > best['cv_score']:
            best = {**run, 'C': C, 'gamma': g, 'kernel': k, 'degree': d, 'cv_score': s}
    return best

best_runs = {
    'Grid Search': recheck(grid_res, 'Grid'),
    'Random Search': recheck(max(rand_runs, key=lambda r: r['cv_score']), 'Random'),
    'PSO': recheck(max(pso_runs, key=lambda r: r['cv_score']), 'PSO')}

### 8. Đánh giá trên tập test
Lấy lần chạy tốt nhất của mỗi phương pháp, huấn luyện lại trên toàn bộ tập train, đánh giá trên tập test. Có thêm baseline SVM mặc định.

In [ ]:
# best_runs = {
#     'Grid Search': grid_res,
#     'Random Search': max(rand_runs, key=lambda r: r['cv_score']),
#     'PSO': max(pso_runs, key=lambda r: r['cv_score'])}

def test_eval(name, C, gamma, kernel, degree):
    pipe = make_pipe(C, gamma, kernel, degree)
    t0 = time.time(); pipe.fit(X_train, y_train); fit_t = time.time() - t0
    pred = pipe.predict(X_test); dec = pipe.decision_function(X_test)
    return {'Method': name, 'C': C, 'gamma': gamma, 'kernel': kernel, 'degree': degree,
            'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred), 'F1': f1_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, dec), 'Final fit (s)': fit_t}, confusion_matrix(y_test, pred)

rows, cms = [], {}
r, cm = test_eval('Default SVM', 1.0, 'scale', 'rbf', 3); rows.append(r); cms['Default SVM'] = cm
for name, res in best_runs.items():
    r, cm = test_eval(name, res['C'], res['gamma'], res['kernel'], res['degree'])
    rows.append(r); cms[name] = cm

test_df = pd.DataFrame(rows)
test_df['Tuning time (s)'] = [np.nan] + [best_runs[m]['time'] for m in best_runs]
test_df.round(4)

In [ ]:
for k, v in cms.items():
    print(k); print(pd.DataFrame(v, index=['True 0', 'True 1'], columns=['Pred 0', 'Pred 1'])); print()

### 9. Trực quan hoá

In [ ]:
colors = {'Grid Search': 'tab:blue', 'Random Search': 'tab:orange', 'PSO': 'tab:green'}

# (a) Đường hội tụ
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(grid_res['history']) + 1), grid_res['history'], label='Grid Search', color=colors['Grid Search'])
for name, runs in [('Random Search', rand_runs), ('PSO', pso_runs)]:
    H = np.array([r['history'] for r in runs]); m, s = H.mean(0), H.std(0)
    x = np.arange(1, H.shape[1] + 1)
    ax.plot(x, m, label=name + f' (mean of {N_RUNS})', color=colors[name])
    ax.fill_between(x, m - s, m + s, alpha=0.2, color=colors[name])
ax.set_xlabel('Số lần đánh giá hàm mục tiêu'); ax.set_ylabel('Best CV-F1 tới thời điểm hiện tại')
ax.set_title('Đường hội tụ (4 siêu tham số)'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('convergence.png', dpi=150); plt.show()

In [ ]:
# (b) Các điểm được duyệt trong mặt phẳng (log C, log gamma); màu = CV-F1, hình = kernel
markers = {0: 'o', 1: '^', 2: 's'}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True, sharey=True)
data = [('Grid Search', grid_res), ('Random Search', best_runs['Random Search']), ('PSO', best_runs['PSO'])]
vmin = min(d['scores'].min() for _, d in data); vmax_ = max(d['scores'].max() for _, d in data)
for ax, (name, d) in zip(axes, data):
    for ki, kn in enumerate(KERNELS):
        m = d['points'][:, 2] == ki
        sc = ax.scatter(d['points'][m, 0], d['points'][m, 1], c=d['scores'][m], cmap='viridis',
                        vmin=vmin, vmax=vmax_, marker=markers[ki], s=45, edgecolor='k', linewidth=0.3, label=kn)
    ax.scatter(np.log10(d['C']), np.log10(d['gamma']), marker='*', s=320, c='red', label='best')
    ax.set_title(name); ax.set_xlabel('log10(C)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('log10(gamma)'); axes[0].legend(fontsize=8)
fig.colorbar(sc, ax=axes, label='CV-F1')
plt.savefig('search_space.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# (c) PSO và Random dồn lượt đánh giá vào kernel nào? (trung bình qua các lần chạy) + CV-F1 tốt nhất theo kernel
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
width = 0.25
for i, (name, runs) in enumerate([('Grid Search', [grid_res]), ('Random Search', rand_runs), ('PSO', pso_runs)]):
    frac = np.mean([[np.mean(r['points'][:, 2] == k) for k in range(len(KERNELS))] for r in runs], axis=0)
    axes[0].bar(np.arange(len(KERNELS)) + i * width, frac, width, label=name, color=colors[name])
axes[0].set_xticks(np.arange(len(KERNELS)) + width); axes[0].set_xticklabels(KERNELS)
axes[0].set_ylabel('Tỉ lệ lượt đánh giá'); axes[0].set_title('Phân bổ lượt đánh giá theo kernel'); axes[0].legend()

for i, (name, runs) in enumerate([('Grid Search', [grid_res]), ('Random Search', rand_runs), ('PSO', pso_runs)]):
    best_k = np.mean([[r['scores'][r['points'][:, 2] == k].max() if (r['points'][:, 2] == k).any() else np.nan
                       for k in range(len(KERNELS))] for r in runs], axis=0)
    axes[1].bar(np.arange(len(KERNELS)) + i * width, best_k, width, label=name, color=colors[name])
axes[1].set_xticks(np.arange(len(KERNELS)) + width); axes[1].set_xticklabels(KERNELS)
axes[1].set_ylabel('CV-F1 tốt nhất'); axes[1].set_title('CV-F1 tốt nhất theo kernel')
plt.tight_layout(); plt.savefig('kernel_analysis.png', dpi=150); plt.show()

In [ ]:
# (d) Chỉ số test và thời gian tuning
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
test_df.set_index('Method')[['Precision', 'Recall', 'F1', 'ROC-AUC']].plot.bar(ax=axes[0], rot=0)
axes[0].set_title('Kết quả trên tập test'); axes[0].set_ylim(0, 1); axes[0].grid(alpha=0.3, axis='y')
tuning_df.set_index('Method')['Time mean (s)'].plot.bar(
    ax=axes[1], rot=0, yerr=tuning_df.set_index('Method')['Time std (s)'],
    color=[colors[m] for m in tuning_df['Method']])
axes[1].set_title('Thời gian tuning trung bình (s)'); axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('test_metrics_time.png', dpi=150); plt.show()

In [ ]:
# (e) Độ ổn định qua các lần chạy
plt.figure(figsize=(6, 4.5))
plt.boxplot([[grid_res['cv_score']], [r['cv_score'] for r in rand_runs], [r['cv_score'] for r in pso_runs]],
            labels=['Grid', 'Random', 'PSO'])
plt.ylabel('Best CV-F1'); plt.title(f'Độ ổn định qua {N_RUNS} lần chạy'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('stability.png', dpi=150); plt.show()

### 10. LƯU KẾT QUẢ

In [ ]:
tuning_df.to_csv('tuning_summary_4params_v2.csv', index=False)
test_df.to_csv('test_results_.csv', index=False)
print(tuning_df.to_string(index=False)); print(); print(test_df.round(4).to_string(index=False))